In [1]:
import pandas as pd
from pathlib import Path
import json
import requests
import time


cwd = Path.cwd()
data_dir = cwd / "data"
dataset_path = data_dir / "World_Important_Data.csv"



# Dataset 1 : World Important Events - Ancient to Modern
## Link: https://www.kaggle.com/datasets/saketk511/world-important-events-ancient-to-modern
### Description:
This dataset, "World Important Events - Ancient to Modern," spans significant historical milestones from ancient times to the modern era, covering diverse global incidents. It provides a comprehensive timeline of events that have shaped the world, offering insights into wars, cultural shifts, technological advancements, and social movements.

#### Column Descriptions:
- **Sl. No**: Serial number.
- **Name of Incident**: Title of the event.
- **Date, Month, Year**: When the event occurred.
- **Country**: Where it happened.
- **Type of Event**: Nature of the event (e.g., War, Revolution).
- **Place Name**: Specific location of the event.
- **Impact**: Brief description of the event's impact.
- **Affected Population**: Who was impacted.
- **Important Person/Group Responsible**: Key figures or groups involved.
- **Outcome**: Result of the event (Positive, Negative, Mixed).



In [39]:
dataset_path = data_dir / "World_Important_Data.csv"
df = pd.read_csv(dataset_path)

In [40]:
df

,Sl. No,Name of Incident,Date,Month,Year,Country,Type of Event,Place Name,Impact,Affected Population,Important Person/Group Responsible,Outcome
0,1,Indus Valley Civilization Flourishes,Unknown,Unknown,2600 BC,India,Civilization,Indus Valley,Development of one of the world's earliest urb...,Local inhabitants,Indus Valley people,Positive
1,2,Battle of the Ten Kings,Unknown,Unknown,1400 BC,India,Battle,Punjab,Rigvedic tribes consolidated their control ove...,Rigvedic tribes,Sudas,Positive
2,6,Establishment of the Delhi Sultanate,Unknown,Unknown,1206,India,Political,Delhi,Muslim rule established in parts of India,People of Delhi and surrounding regions,QutbUnknownudUnknowndin Aibak,Mixed
3,7,Battle of Panipat,21,April,1526,India,Battle,Panipat,Foundation of the Mughal Empire in India,Northern Indian kingdoms,Babur,Mixed
4,8,Establishment of British Raj,1,May,1858,India,Colonial,Whole India,Start of direct British governance in India,Indian subcontinent,British East India Company/Empire,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...
1091,1147,First Mexican Empire Declared,28,September,1821,Mexico,Political,Mexico,Brief establishment of an empire soon transiti...,Mexicans,Agustín de Iturbide,Positive
1092,1148,U.S.UnknownMexican War,25,April,1846,Mexico,Military,Northern Mexico,Loss of vast territories to the United States,Mexicans,US,Negative
1093,1149,Reform Wars,Unknown,Unknown,1857,Mexico,Civil War,Mexico,Liberal vs. Conservative conflict leading to c...,Mexicans,Benito Juárez,Mixed
1094,1150,French Intervention in Mexico,Unknown,Unknown,1862,Mexico,Military Intervention,Mexico,Establishment and fall of the Second Mexican E...,Mexicans,Napoleon III,Negative


## Cleaning Dataset 1

Dataset1 for the most part contains inforamtion relvenet to data on histotical events.The only column that is not relevant to our project is the outcome column as judging wheter an outcome was postive or negative is not relevant to our project as we are only interested in historical events.
## Dataset Columns after cleaning:
- **Name**: Name of the historical figure (in English).
- **en_curid**: Unique identifier for each individual; maps to the Wikipedia page ID and can be used to access the biography via `http://en.wikipedia.org/?curid=[en_curid]`.
- **Birth Year**: Year the individual was born.
- **Birth City**: City where the individual was born.
- **Country Name**: Commonly accepted name of the country.
- **Continent Name**: Name of the continent associated with the individual.
- **Occupation**: Specific occupation of the individual.
- **Industry**: Broader category grouping related occupations.
- **Domain**: Higher-level category grouping related industries.
- **Gender**: Gender of the individual (Male/Female).


In [41]:
df = df.drop(columns="Outcome")

In [42]:
df = df.rename(columns={
    "Sl. No": "id",
    "Name of Incident": "event_name",
    "Date": "day",
    "Month": "month",
    "Year": "year",
    "Country": "country",
    "Type of Event": "event_type",
    "Place Name": "place_name",
    "Impact": "impact",
    "Affected Population": "affected_population",
    "Important Person/Group Responsible": "responsible_entity"
})

In [43]:
df.columns


Index(['id', 'event_name', 'day', 'month', 'year', 'country', 'event_type',
       'place_name', 'impact', 'affected_population', 'responsible_entity'],
      dtype='str')

## Verifying uniqueness of each row for Dataset 1


In [49]:
df.duplicated().sum()


np.int64(0)

In [50]:
df["id"].nunique(), len(df)


(1096, 1096)

## Cleaning Dates

In [44]:
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}


In [45]:
df = df.replace("Unknown", pd.NA)

In [46]:
df["month"] = df["month"].map(month_map)
df["month"] = df["month"].astype("Int64")

## Handle BC years

In [47]:
def parse_year(val):
    if pd.isna(val):
        return pd.NA
    val = str(val).strip()

    if "BC" in val:
        return -int(val.replace("BC", "").strip())
    else:
        return int(val)

## Handle Unknown Day Values
Some of day values contain strange values that are not able to converted into ints
ie : 23Unknown25

For these we converted to just NAN and also kept original value. The NAN value will be used by the SQL DB to query and the original value can be used by the LLM for added context.


In [48]:
def clean_day_value(val):
    if pd.isna(val):
        return pd.NA
    val = str(val).strip()
    if val.isdigit():
        day = int(val)
        if 1 <= day <= 31:
            return day
    return pd.NA

In [49]:
df["raw_day"] = df["day"].astype("string")
df["day"] = df["day"].apply(clean_day_value).astype("Int64")

In [50]:
df["year"] = df["year"].apply(parse_year).astype("Int64")

In [51]:
df

,id,event_name,day,month,year,country,event_type,place_name,impact,affected_population,responsible_entity,raw_day
0,1,Indus Valley Civilization Flourishes,<NA>,<NA>,-2600,India,Civilization,Indus Valley,Development of one of the world's earliest urb...,Local inhabitants,Indus Valley people,<NA>
1,2,Battle of the Ten Kings,<NA>,<NA>,-1400,India,Battle,Punjab,Rigvedic tribes consolidated their control ove...,Rigvedic tribes,Sudas,<NA>
2,6,Establishment of the Delhi Sultanate,<NA>,<NA>,1206,India,Political,Delhi,Muslim rule established in parts of India,People of Delhi and surrounding regions,QutbUnknownudUnknowndin Aibak,<NA>
3,7,Battle of Panipat,21,4,1526,India,Battle,Panipat,Foundation of the Mughal Empire in India,Northern Indian kingdoms,Babur,21
4,8,Establishment of British Raj,1,5,1858,India,Colonial,Whole India,Start of direct British governance in India,Indian subcontinent,British East India Company/Empire,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1091,1147,First Mexican Empire Declared,28,9,1821,Mexico,Political,Mexico,Brief establishment of an empire soon transiti...,Mexicans,Agustín de Iturbide,28
1092,1148,U.S.UnknownMexican War,25,4,1846,Mexico,Military,Northern Mexico,Loss of vast territories to the United States,Mexicans,US,25
1093,1149,Reform Wars,<NA>,<NA>,1857,Mexico,Civil War,Mexico,Liberal vs. Conservative conflict leading to c...,Mexicans,Benito Juárez,<NA>
1094,1150,French Intervention in Mexico,<NA>,<NA>,1862,Mexico,Military Intervention,Mexico,Establishment and fall of the Second Mexican E...,Mexicans,Napoleon III,<NA>


In [52]:
df.columns

Index(['id', 'event_name', 'day', 'month', 'year', 'country', 'event_type',
       'place_name', 'impact', 'affected_population', 'responsible_entity',
       'raw_day'],
      dtype='str')

## Verify Datatypes

In [53]:
print(df.dtypes)

id                      int64
event_name                str
day                     Int64
month                   Int64
year                    Int64
country                   str
event_type                str
place_name                str
impact                    str
affected_population       str
responsible_entity        str
raw_day                string
dtype: object


# Dataset 2 : A Manually Verified Dataset of Globally Famous Biographies
## Link: https://www.kaggle.com/datasets/saketk511/world-important-events-ancient-to-modern
### Description:
We present the Pantheon 1.0 dataset: a manually verified dataset of individuals that have transcended linguistic, temporal, and geographic boundaries. The Pantheon 1.0 dataset includes the 11,341 biographies present in more than 25 languages in Wikipedia and is enriched with: (i) manually verified demographic information (place and date of birth, gender) (ii) a taxonomy of occupations classifying each biography at three levels of aggregation and (iii) two measures of global popularity including the number of languages in which a biography is present in Wikipedia (L), and the Historical Popularity Index (HPI) a metric that combines information on L, time since birth, and page-views (2008-2013). We compare the Pantheon 1.0 dataset to data from the 2003 book, Human Accomplishments, and also to external measures of accomplishment in individual games and sports: Tennis, Swimming, Car Racing, and Chess. In all of these cases we find that measures of popularity (L and HPI) correlate highly with individual accomplishment, suggesting that measures of global popularity proxy the historical impact of individuals.

#### Column Descriptions:
- **Name**: Name of the historical figure (in English).
- **en_curid**: Unique identifier for each individual; maps to the Wikipedia page ID and can be used to access the biography via `http://en.wikipedia.org/?curid=[en_curid]`.
- **Country Code (Alpha-2)**: ISO 3166-1 alpha-2 country code based on present-day political boundaries.
- **Country Code (Alpha-3)**: ISO 3166-1 alpha-3 country code based on present-day political boundaries.
- **Country Name**: Commonly accepted name of the country.
- **Continent Name**: Name of the continent associated with the individual.
- **Birth Year**: Year the individual was born.
- **Birth City**: City where the individual was born.
- **Occupation**: Specific occupation of the individual.
- **Industry**: Broader category grouping related occupations.
- **Domain**: Higher-level category grouping related industries.
- **Gender**: Gender of the individual (Male/Female).
- **Total Page Views**: Total Wikipedia page views across all language editions (Jan 2008 – Dec 2013).
- **L\***: Adjusted popularity metric (see dataset appendix for calculation).
- **Number of Languages**: Number of Wikipedia language editions where the biography appears (as of May 2013).
- **Std Page Views**: Standard deviation of page views over time (Jan 2008 – Dec 2013).
- **Page Views (English)**: Total views from the English Wikipedia (Jan 2008 – Dec 2013).
- **Page Views (Non-English)**: Total views from all non-English Wikipedias (Jan 2008 – Dec 2013).
- **Average Views**: Average page views per language edition.
- **HPI (Historical Popularity Index)**: Composite metric measuring historical prominence (see equation (4) in dataset documentation).


In [17]:
dataset_path2 = data_dir / "dataverse_files" / "pantheon.tsv"
df2 = pd.read_csv(dataset_path2, sep="\t")

In [18]:
df2

,en_curid,name,numlangs,birthcity,birthstate,countryName,countryCode,countryCode3,LAT,LON,...,occupation,industry,domain,TotalPageViews,L_star,StdDevPageViews,PageViewsEnglish,PageViewsNonEnglish,AverageViews,HPI
0,307,Abraham Lincoln,131,Hodgenville,KY,UNITED STATES,US,USA,37.571111,-85.738611,...,POLITICIAN,GOVERNMENT,INSTITUTIONS,66145211,5.801387,586914.722000,41477236,24667975,504925.27480,27.938585
1,308,Aristotle,152,Stageira,NaN,Greece,GR,GRC,40.333333,23.500000,...,PHILOSOPHER,PHILOSOPHY,HUMANITIES,56355172,11.914597,201067.460700,15745351,40609821,370757.71050,31.993795
2,339,Ayn Rand,55,Saint Petersburg,NaN,Russia,RU,RUS,59.950000,30.300000,...,WRITER,LANGUAGE,HUMANITIES,14208218,3.175685,87632.490200,11023490,3184728,258331.23640,24.325936
3,595,Andre Agassi,69,Las Vegas,NV,UNITED STATES,US,USA,36.121514,-115.173851,...,TENNIS PLAYER,INDIVIDUAL SPORTS,SPORTS,11244030,6.242525,85553.318100,6353888,4890142,162956.95650,20.925999
4,628,Aldous Huxley,62,Godalming,NaN,UNITED KINGDOM,GB,GBR,51.185000,-0.610000,...,WRITER,LANGUAGE,HUMANITIES,9268920,6.219842,33037.032090,5137256,4131664,149498.70970,25.996605
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11336,38203876,Rebecca,32,Other,NaN,Unknown,UNK,UNK,NaN,NaN,...,RELIGIOUS FIGURE,RELIGION,INSTITUTIONS,1456125,3.958282,3607.911055,954112,502013,45503.90625,25.905977
11337,38239735,Mindaugas,31,Lithuania,NaN,Lithuania,LT,LTU,55.000000,24.000000,...,POLITICIAN,GOVERNMENT,INSTITUTIONS,814092,7.973357,3304.418133,164031,650061,26261.03226,24.958617
11338,38268805,Gediminas of Lithuania,31,Other,NaN,Unknown,UNK,UNK,NaN,NaN,...,POLITICIAN,GOVERNMENT,INSTITUTIONS,737870,9.070608,3047.999702,132408,605462,23802.25806,24.929128
11339,38646961,Eric Hobsbawm,49,Alexandria,NaN,Egypt,EG,EGY,31.200000,29.916667,...,HISTORIAN,HISTORY,HUMANITIES,2557682,7.711120,40990.920320,952975,1604707,52197.59184,23.372257


In [19]:
df2.columns

Index(['en_curid', 'name', 'numlangs', 'birthcity', 'birthstate',
       'countryName', 'countryCode', 'countryCode3', 'LAT', 'LON',
       'continentName', 'birthyear', 'gender', 'occupation', 'industry',
       'domain', 'TotalPageViews', 'L_star', 'StdDevPageViews',
       'PageViewsEnglish', 'PageViewsNonEnglish', 'AverageViews', 'HPI'],
      dtype='str')

## Cleaning Dataset 2

Dataset currently has analytical data about wikipedia information that may not be relevant to our project. Although useful for research, analysis these are not useful for our project. We will focus on the dataset that contains information about historical events.

## Dataset Columns after cleaning:
- **Name**: Name of the historical figure (in English).
- **en_curid**: Unique identifier for each individual; maps to the Wikipedia page ID and can be used to access the biography via `http://en.wikipedia.org/?curid=[en_curid]`.
- **Birth Year**: Year the individual was born.
- **Birth City**: City where the individual was born.
- **Country Name**: Commonly accepted name of the country.
- **Continent Name**: Name of the continent associated with the individual.
- **Occupation**: Specific occupation of the individual.
- **Industry**: Broader category grouping related occupations.
- **Domain**: Higher-level category grouping related industries.
- **Gender**: Gender of the individual (Male/Female).

In [20]:
cols_to_keep = [
    "name",
    "en_curid",
    "birthyear",
    "birthcity",
    "countryName",
    "continentName",
    "occupation",
    "industry",
    "domain",
    "gender"
]

df2 = df2[cols_to_keep]

In [21]:
df2 = df2.rename(columns={
    "name": "name",
    "en_curid": "curid",
    "birthyear": "birth_year",
    "birthcity": "birth_city",
    "countryName": "country",
    "continentName": "continent",
    "occupation": "occupation",
    "industry": "industry",
    "domain": "domain",
    "gender": "gender"
})

In [22]:
df2

,name,curid,birth_year,birth_city,country,continent,occupation,industry,domain,gender
0,Abraham Lincoln,307,1809,Hodgenville,UNITED STATES,North America,POLITICIAN,GOVERNMENT,INSTITUTIONS,Male
1,Aristotle,308,-384,Stageira,Greece,Europe,PHILOSOPHER,PHILOSOPHY,HUMANITIES,Male
2,Ayn Rand,339,1905,Saint Petersburg,Russia,Europe,WRITER,LANGUAGE,HUMANITIES,Female
3,Andre Agassi,595,1970,Las Vegas,UNITED STATES,North America,TENNIS PLAYER,INDIVIDUAL SPORTS,SPORTS,Male
4,Aldous Huxley,628,1894,Godalming,UNITED KINGDOM,Europe,WRITER,LANGUAGE,HUMANITIES,Male
...,...,...,...,...,...,...,...,...,...,...
11336,Rebecca,38203876,-3500,Other,Unknown,Unknown,RELIGIOUS FIGURE,RELIGION,INSTITUTIONS,Female
11337,Mindaugas,38239735,1200,Lithuania,Lithuania,Europe,POLITICIAN,GOVERNMENT,INSTITUTIONS,Male
11338,Gediminas of Lithuania,38268805,1275,Other,Unknown,Unknown,POLITICIAN,GOVERNMENT,INSTITUTIONS,Male
11339,Eric Hobsbawm,38646961,1917,Alexandria,Egypt,Africa,HISTORIAN,HISTORY,HUMANITIES,Male


## Verifying uniqueness of each row for Dataset 2

In [23]:
df2.duplicated().sum()

np.int64(0)

In [24]:
df2["curid"].nunique(), len(df2)

(11341, 11341)

## Cleaning Dates

In [25]:
df2 = df2.replace("Unknown", pd.NA)

In [26]:
df2["birth_year"] = pd.to_numeric(
    df2["birth_year"],
    errors="coerce"
).astype("Int64")

In [27]:
df2

,name,curid,birth_year,birth_city,country,continent,occupation,industry,domain,gender
0,Abraham Lincoln,307,1809,Hodgenville,UNITED STATES,North America,POLITICIAN,GOVERNMENT,INSTITUTIONS,Male
1,Aristotle,308,-384,Stageira,Greece,Europe,PHILOSOPHER,PHILOSOPHY,HUMANITIES,Male
2,Ayn Rand,339,1905,Saint Petersburg,Russia,Europe,WRITER,LANGUAGE,HUMANITIES,Female
3,Andre Agassi,595,1970,Las Vegas,UNITED STATES,North America,TENNIS PLAYER,INDIVIDUAL SPORTS,SPORTS,Male
4,Aldous Huxley,628,1894,Godalming,UNITED KINGDOM,Europe,WRITER,LANGUAGE,HUMANITIES,Male
...,...,...,...,...,...,...,...,...,...,...
11336,Rebecca,38203876,-3500,Other,NaN,NaN,RELIGIOUS FIGURE,RELIGION,INSTITUTIONS,Female
11337,Mindaugas,38239735,1200,Lithuania,Lithuania,Europe,POLITICIAN,GOVERNMENT,INSTITUTIONS,Male
11338,Gediminas of Lithuania,38268805,1275,Other,NaN,NaN,POLITICIAN,GOVERNMENT,INSTITUTIONS,Male
11339,Eric Hobsbawm,38646961,1917,Alexandria,Egypt,Africa,HISTORIAN,HISTORY,HUMANITIES,Male


## Verify Datatypes
It is important to verify that the data types are appropiate as they will be converted into SQL types

In [36]:
print(df2.dtypes)


name            str
curid         int64
birth_year    Int64
birth_city      str
country         str
continent       str
occupation      str
industry        str
domain          str
gender          str
dtype: object


## QA pairing dataset formatting

In [3]:

def load_qa_json(json_path, time_period):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data["qa_pairs"])
    df["time_period"] = time_period

    df = df.reset_index(drop=True)
    df["id"] = df.index + 1

    return df[["id", "question", "answer", "time_period"]]



# Dataset 3: World History to 1500: Q&A Dataset
## Link: https://github.com/provos/world-history-to-1500-qa
### Description:
This dataset consists of high-quality question and answer pairs generated from the content of "World History Volume 1: to 1500". The Q&A pairs are designed to cover key historical events, figures, and concepts from prehistory to 1500 CE, providing a comprehensive resource for students, educators, and history enthusiasts.

# Size of Dataset:
2 Columns x 1757 Rows


#### Column Descriptions:
- **Question**: Natural language question related to world history, derived from textbook content. Covers topics such as historical events, figures, civilizations, and key developments within the dataset’s time period.
- **Answer**: Detailed explanatory response to the corresponding question. Provides factual information, contextual background, and historical significance based on the source textbook.


In [4]:
dataset_path3 = data_dir / "WH_QA1.json"
time_period = 'prehistory_to_1500'

In [6]:
df_qa1 = load_qa_json(
    dataset_path3,
    time_period="prehistory_to_1500"
)

In [ ]:
df_qa1.to_json(
    "qa1_clean.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# Dataset 4: World History Since 1500: Q&A Dataset
## Link: https://github.com/provos/world-history-to-1500-qa
### Description:
This dataset consists of high-quality question and answer pairs generated from the content of "World History Since 1500: An Open and Free Textbook". The Q&A pairs are designed to cover key historical events, figures, and concepts from 1500 to the present day, providing a comprehensive resource for students, educators, and history enthusiasts.

# Size of Dataset:
2 Columns x 376 Rows

#### Column Descriptions:
- **Question**: Natural language question related to world history, derived from textbook content. Covers topics such as historical events, figures, civilizations, and key developments within the dataset’s time period.
- **Answer**: Detailed explanatory response to the corresponding question. Provides factual information, contextual background, and historical significance based on the source textbook.




In [8]:
dataset_path4 = data_dir / "WH_QA2.json"
time_period4 = 'history_after_1500'

In [9]:
df_qa2 = load_qa_json(
    dataset_path4,
    time_period=time_period4
)

In [10]:
df_qa2.to_json(
    "qa2_clean.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

## Saving Datasets
To help project strucuture clean and organized this section where I cleaned the data is a seperate
part from the postgres upload to Neon. With that being said the upload will be done in the file :


In [54]:
df.to_csv("historical_events_core.csv", index=False)

In [38]:
df2.to_csv("historical_figures.csv", index=False)

## Extracting Biography Data from Wikipedia

In [ ]:
def get_wikipedia_text(curid):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "pageids": curid,
        "format": "json"
    }
    res = requests.get(url, params=params).json()
    pages = res["query"]["pages"]
    return list(pages.values())[0].get("extract", "")

## Testing Text Data extraction from Wikipedia for specific curid
Ayn Rand = 339
Jane Adams = 152277


In [64]:
import requests

def get_wikipedia_text(curid):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "pageids": curid,
        "format": "json"
    }

    headers = {
        "User-Agent": "HistLLM-Research/1.0 (SDSU Graduate Research Project; aespinozamerid6680@sdsu.edu)"
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=20)
        response.raise_for_status()

        data = response.json()

        pages = data.get("query", {}).get("pages", {})
        if not pages:
            return ""

        page = list(pages.values())[0]
        return page.get("extract", "")

    except Exception as e:
        print(f"❌ Error fetching curid {curid}: {e}")
        return ""

In [ ]:
# print(get_wikipedia_text(339))

In [ ]:
# df5 = df2[["curid", "name"]].copy()

In [68]:


def fetch_biographies(
    df,
    save_every=100,
    sleep_seconds=0.1,
    output_csv="biographies_progress.csv"
):
    biographies = []
    output_path = Path(output_csv)

    total = len(df)
    start_time = time.time()

    # Resume support
    done_ids = set()
    if output_path.exists():
        existing = pd.read_csv(output_path)
        if "curid" in existing.columns:
            done_ids = set(existing["curid"].dropna().astype("Int64").tolist())
            print(f"Found existing progress file with {len(done_ids)} completed rows.")

    remaining_df = df[~df["curid"].isin(done_ids)].copy()
    remaining_total = len(remaining_df)

    print(f"Starting biography fetch.")
    print(f"Total input rows: {total}")
    print(f"Remaining rows to process: {remaining_total}")
    print(f"Saving progress every {save_every} rows.")
    print("-" * 60)

    processed_since_save = 0
    processed_total = 0

    for idx, row in enumerate(remaining_df.itertuples(index=False), start=1):
        curid = row.curid
        name = row.name

        try:
            bio = get_wikipedia_text(curid)
        except Exception as e:
            print(f"[ERROR] curid={curid}, name={name}: {e}")
            bio = ""

        if not bio or str(bio).strip() == "":
            print(f"[WARNING] Missing bio for {name} ({curid})")
            bio = "Biography not available."


        biographies.append({
            "curid": curid,
            "name": name,
            "biography": bio
        })

        processed_since_save += 1
        processed_total += 1

        if processed_total % 10 == 0 or processed_total == 1:
            elapsed = time.time() - start_time
            rate = processed_total / elapsed if elapsed > 0 else 0
            eta = (remaining_total - processed_total) / rate if rate > 0 else float("inf")
            print(
                f"[PROGRESS] {processed_total}/{remaining_total} "
                f"({processed_total/remaining_total:.1%}) | "
                f"curid={curid} | name={name} | "
                f"elapsed={elapsed/60:.1f} min | eta={eta/60:.1f} min"
            )

        if processed_since_save >= save_every:
            chunk_df = pd.DataFrame(biographies)

            if output_path.exists():
                chunk_df.to_csv(output_path, mode="a", header=False, index=False)
            else:
                chunk_df.to_csv(output_path, index=False)

            print(f"[SAVE] Wrote {len(chunk_df)} rows to {output_csv}")
            biographies.clear()
            processed_since_save = 0

        time.sleep(sleep_seconds)

    # Final flush
    if biographies:
        chunk_df = pd.DataFrame(biographies)
        if output_path.exists():
            chunk_df.to_csv(output_path, mode="a", header=False, index=False)
        else:
            chunk_df.to_csv(output_path, index=False)

        print(f"[FINAL SAVE] Wrote final {len(chunk_df)} rows to {output_csv}")

    total_elapsed = time.time() - start_time
    print("-" * 60)
    print(f"Done. Total processed this run: {processed_total}")
    print(f"Total elapsed: {total_elapsed/60:.2f} minutes")

    #Clean UP NAN Biogrpahies


    return pd.read_csv(output_path)

In [ ]:
fetch_biographies(df2,output_csv="biographies.csv")